# 🧩 Mini-Lab: Structured Logging

**Module: Trust but Verify: Validation, Config, Errors, Logs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why structured (JSON) logs are superior to plain-text `print()` statements for debugging and monitoring
2. **Emit** structured log records with a run ID, timestamps, and per-record fields using Python's built-in `logging` module
3. **Redact** sensitive data (API keys and user text) in log output to maintain secrets hygiene

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Structured Logging | Logs as records with a run id and consistent per-record fields, so you can trace behavior and search reliably, while redacting sensitive keys and user text by default. |
| Secrets Hygiene | Secrets hygiene is the habit of keeping passwords, API keys, and tokens out of code and logs, sharing them safely, and rotating them regularly so one mistake doesn’t become a breach. |


## Why Structured Logging?

Plain-text logs like `print("Processing item 42...")` are easy to write but hard to search, compare, or trust. Structured logging turns each log line into a JSON record that tools and humans can both read reliably.

- **Consistent fields**: every record has the same keys (`timestamp`, `level`, `message`, ...)
- **Easy filtering**: find everything from one run or one item by `run_id` or `item_id`
- **Safer by default**: a redaction filter masks secrets and user text before they are written

> 🧩 **Structured logging.** The run-id records from the video, built here step by step.

In [1]:
import json
import logging
import uuid
from datetime import datetime, timezone


class JSONFormatter(logging.Formatter):
    """Format every log record as a single JSON line."""

    def format(self, record: logging.LogRecord) -> str:
        log_entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "message": record.getMessage(),
        }
        # Merge any extra fields passed via `extra={}`
        # (api_key and user_text are optional fields that the redaction filter cleans later)
        for key in ("run_id", "item_id", "model", "tokens", "latency_ms",
                    "api_key", "user_text"):
            if hasattr(record, key):
                log_entry[key] = getattr(record, key)
        return json.dumps(log_entry)

In [2]:
def get_logger(name: str = "app") -> logging.Logger:
    """Create (or retrieve) a logger that outputs JSON lines."""
    logger = logging.getLogger(name)

    # Avoid adding duplicate handlers on re-runs in a notebook
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(JSONFormatter())
        logger.addHandler(handler)
        logger.setLevel(logging.DEBUG)

    return logger


# Generate a run_id for this execution
run_id = uuid.uuid4().hex[:8]
logger = get_logger()

logger.info("Pipeline started", extra={"run_id": run_id})

{"timestamp": "2026-10-18T20:20:43.227548+00:00", "level": "INFO", "message": "Pipeline started", "run_id": "b1d7dd87"}


In [3]:
import time

# Simulate processing a couple of items
items = [
    {"id": "doc-001", "text": "Summarize the quarterly earnings report."},
    {"id": "doc-002", "text": "Translate this paragraph to French."},
]

for item in items:
    start = time.monotonic()
    time.sleep(0.05)  # stand-in for a model call
    latency = (time.monotonic() - start) * 1000

    # Per-record context fields go in `extra`, not in the message string
    logger.info(
        "Item processed",
        extra={
            "run_id": run_id,
            "item_id": item["id"],
            "model": "gpt-4o-mini",
            "tokens": 124,
            "latency_ms": round(latency, 1),
        },
    )

{"timestamp": "2026-10-18T20:20:43.290738+00:00", "level": "INFO", "message": "Item processed", "run_id": "b1d7dd87", "item_id": "doc-001", "model": "gpt-4o-mini", "tokens": 124, "latency_ms": 47.0}


{"timestamp": "2026-10-18T20:20:43.342203+00:00", "level": "INFO", "message": "Item processed", "run_id": "b1d7dd87", "item_id": "doc-002", "model": "gpt-4o-mini", "tokens": 124, "latency_ms": 47.0}


## Redact Secrets and User Text

It is easy to log a full API key or a user's private input by accident. A redaction filter cleans every record before it is written:

- **API keys**: strings like `sk-...` are masked
- **User text**: truncated to a short preview

In [4]:
import re


class RedactingFilter(logging.Filter):
    """Redact API keys and truncate user text in log records."""

    # Matches common API key patterns
    KEY_PATTERN = re.compile(r"(sk-|key-|token-)[A-Za-z0-9_-]{4,}")
    MAX_USER_TEXT = 30  # characters to keep before truncating

    def filter(self, record: logging.LogRecord) -> bool:
        # Mask API keys in the message, keeping only a short prefix
        record.msg = self.KEY_PATTERN.sub(
            lambda m: m.group()[:6] + "...REDACTED", str(record.msg)
        )
        if hasattr(record, "api_key"):
            record.api_key = str(record.api_key)[:6] + "...REDACTED"

        # Keep only a short preview of user text
        if hasattr(record, "user_text"):
            text = str(record.user_text)
            if len(text) > self.MAX_USER_TEXT:
                record.user_text = text[: self.MAX_USER_TEXT] + "…[truncated]"

        return True  # always keep the record, just cleaned


# Attach the filter to our logger
logger.addFilter(RedactingFilter())

In [5]:
# See redaction in action: the key is masked and the user text is truncated
logger.info(
    "Connecting with key sk-abc123realkey789xyz",
    extra={"run_id": run_id},
)

long_user_input = "Please summarize this confidential document about our Q4 merger plans."

logger.info(
    "Processing user request",
    extra={
        "run_id": run_id,
        "item_id": "req-042",
        "user_text": long_user_input,
    },
)

{"timestamp": "2026-10-18T20:20:43.378754+00:00", "level": "INFO", "message": "Connecting with key sk-abc...REDACTED", "run_id": "b1d7dd87"}


{"timestamp": "2026-10-18T20:20:43.380755+00:00", "level": "INFO", "message": "Processing user request", "run_id": "b1d7dd87", "item_id": "req-042", "user_text": "Please summarize this confiden\u2026[truncated]"}


## 🎯 Summary

### Key Takeaways

1. **Structured logging** — emit log records as JSON objects with consistent fields (`timestamp`, `level`, `run_id`, `item_id`) so logs are machine-parseable and filterable, not free-form text
2. **Run ID correlation** — generate a unique `run_id` per execution and attach it to every log record so you can trace all events from a single pipeline run
3. **Per-record context** — pass contextual fields (model, tokens, latency) via `extra={}` to make each log line self-describing without string interpolation
4. **Redaction filter** — attach a `logging.Filter` that automatically masks API keys and truncates user text before records are written, so secrets never leak into log output